# MVO API：受约束优化与可微优化层

本 Notebook 从一个可解释的小例子介绍 NumPy、CVXPY、PyTorch 与 CVXPYlayers 的组合使用，再检查项目的公开接口。

目标为 $\min_w\;\gamma w^T\Sigma w-\mu^Tw$，约束 $w\geq0,\;\mathbf 1^Tw=1$。所有均值和协方差都按年计量，$\gamma=5$。

可选换手约束为 $\|w-w_{prev}\|_1\leq\tau$。这里 L1 换手表示买入金额与卖出金额之和，$\tau=0.4$ 相当于最多卖出20%并买入20%。

参考：[CVXPY](https://www.cvxpy.org/)、[CVXPYlayers](https://github.com/cvxpy/cvxpylayers)、[项目 API 文档](mvo.API.md)。

核心实现见 [mvo_utils.ipynb](mvo_utils.ipynb)。开头的 `%run` 会载入其中的函数；修改后请保存，再重启内核并从头运行本 Notebook。

In [1]:
# 载入核心代码 Notebook；其中包含固定版本的依赖安装。
%run ./mvo_utils.ipynb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import torch
import cvxpy as cp
from IPython.display import display

torch.set_num_threads(1)
print("可用求解器：", cp.installed_solvers())
assert "CLARABEL" in cp.installed_solvers()

可用求解器： ['CLARABEL', 'SCS', 'DIFFCP', 'SCIPY', 'HIGHS', 'OSQP']


## 1. 使用 CVXPY 表达优化问题

三个假设资产的年化收益是 8%、12%、6%，波动率是 20%、30%、40%，假设它们不相关。这些数据用于 API 演示，不能作为市场预测。

In [3]:
mean = np.array([0.08, 0.12, 0.06])
covariance = np.diag([0.04, 0.09, 0.16])
w = cp.Variable(3)
problem = cp.Problem(cp.Minimize(5 * cp.quad_form(w, covariance) - mean @ w),
                     [w >= 0, cp.sum(w) == 1])
problem.solve(solver=cp.CLARABEL)
solver = MeanVarianceSolver(3)
weights = solver.solve(mean, covariance)
np.testing.assert_allclose(weights, w.value, atol=1e-5)
display(pd.DataFrame({"asset": ["A", "B", "C"], "weight": weights}))
assert np.isclose(weights.sum(), 1)
assert (weights >= 0).all()

,asset,weight
0,A,0.571311
1,B,0.298361
2,C,0.130328


## 2. 用解析解检查最小方差

零预期收益、对角协方差时，最小方差组合的权重与方差的倒数成正比。它为求解器提供独立的正确性检查。

In [4]:
minimum_variance = solver.solve(np.zeros(3), covariance)
expected = 1 / np.diag(covariance)
expected /= expected.sum()
np.testing.assert_allclose(minimum_variance, expected, atol=1e-5)
print("解析解检查通过：", minimum_variance)

解析解检查通过： [0.59016393 0.26229508 0.14754099]


## 3. 协方差因子与可微层

令 $F^TF=\Sigma$，于是 $w^T\Sigma w=\|Fw\|^2$。这种参数化让 CVXPYlayers 可以对输入均值和因子求导。

对第一个权重反向传播只是梯度检查。完整训练使用实际持有期的均值—方差效用。

In [5]:
factor = covariance_factor(covariance)
np.testing.assert_allclose(factor.T @ factor, covariance)
mean_tensor = torch.tensor(mean, dtype=torch.float64, requires_grad=True)
factor_tensor = torch.tensor(factor, dtype=torch.float64)
layer = make_allocation_layer(3)
layer_weights, = layer(mean_tensor, factor_tensor, solver_args={"eps": 1e-8})
layer_weights[0].backward()
np.testing.assert_allclose(layer_weights.detach().numpy(), weights, atol=2e-4)
assert torch.isfinite(mean_tensor.grad).all()
assert mean_tensor.grad.abs().sum().item() > 1e-4
print("权重：", layer_weights.detach().numpy())
print("第一个权重对均值的梯度：", mean_tensor.grad.numpy())

权重： [0.57131147 0.29836066 0.13032787]
第一个权重对均值的梯度： [ 1.02459016 -0.6557377  -0.36885246]


## 4. 加入换手约束

上一期已投资权重是输入的一部分；普通求解器和可微层共享相同的问题定义。初始现金配置属于回测记账，这个接口的 `previous` 必须是满仓的历史权重。

In [6]:
previous = np.array([0.1, 0.2, 0.7])
limit = 0.4
bounded = MeanVarianceSolver(3, turnover_limit=limit).solve(mean, covariance, previous)
bounded_layer = make_allocation_layer(3, turnover_limit=limit)
differentiable, = bounded_layer(
    mean_tensor.detach(), factor_tensor, torch.tensor(previous), solver_args={"eps": 1e-8}
)
np.testing.assert_allclose(differentiable.numpy(), bounded, atol=2e-4)
assert np.abs(bounded - previous).sum() <= limit + 2e-4
display(pd.DataFrame({"previous": previous, "unconstrained_turnover": weights,
                      "turnover_limited": bounded}))

,previous,unconstrained_turnover,turnover_limited
0,0.1,0.571311,0.3
1,0.2,0.298361,0.2
2,0.7,0.130328,0.5


## 5. 从历史特征到训练接口

下面使用离线模拟数据检查 API：特征和协方差只使用配置日之前的数据，训练/验证持有期互不重叠。两轮训练仅用于演示接口，不用于比较策略优劣。

In [7]:
samples = make_samples(synthetic_returns(days=450, assets=3))
keys = ("features", "factors", "future")
train = tuple(samples[key][:8] for key in keys)
validation = tuple(samples[key][8:12] for key in keys)
model = train_model(train, validation, "utility", epochs=2, seed=7)
predicted_mean = model.predict_mean(samples["features"][12])
F = samples["factors"][12]
print("预测均值：", predicted_mean)
print("组合权重：", solver.solve(predicted_mean, F.T @ F))
print("验证选模信息：", model.metadata)
print("API Notebook passed.")

预测均值： [0.12465455 0.07948314 0.05088447]
组合权重： [0.56487328 0.18096471 0.25416201]
验证选模信息： {'mode': 'utility', 'seed': 7, 'epochs': 2, 'best_epoch': 2, 'validation_loss': -0.13020448544521723, 'train_seconds': 0.32151354199959314, 'training_samples': 8, 'validation_samples': 4}
API Notebook passed.
